# NB10D — pHash retrieval + guarded ECC consensus verifier (EXPERIMENT)

Mục tiêu của nhánh này là xử lý đúng failure mode thấy ở NB10A/NB10C: nhiều **same-image / near-identical** vẫn rơi vào manual do resize, placement, JPEG hoặc brightness nhỏ; đồng thời không để cùng form nhưng khác màu/detail bị auto-duplicate.

Pipeline:

`grayscale pHash retrieval -> RGB foreground normalize -> guarded Euclidean ECC -> multi-evidence consensus -> DUP / MANUAL / NON`

ECC ở đây chỉ cho **translation + rotation nhỏ**. Không dùng affine scale/shear/homography vì verifier không được phép bóp méo một garment khác thành giống garment còn lại.

Auto-DUP cần đồng thời qua nhiều bằng chứng: ECC correlation, RGB SSIM, foreground IoU, edge SSIM, Lab color distance, interior MAE và worst-patch MAE. Threshold hiện vẫn **EXPERIMENTAL_UNCALIBRATED**.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

# Optional: use a Colab Secret named HF_TOKEN if available.
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
BRANCH = 'feat/evaluation3-ecc-neardup-verifier'
REPO_ROOT = Path('/content/opisoverated-e3-ecc-verifier')

def run_git(*args, cwd=None):
    return subprocess.run(['git', '-c', 'http.version=HTTP/1.1', *args], cwd=cwd, check=True, text=True)

if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_git('clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT))
else:
    run_git('fetch', 'origin', BRANCH, cwd=REPO_ROOT)
    run_git('switch', BRANCH, cwd=REPO_ROOT)
    run_git('pull', '--ff-only', 'origin', BRANCH, cwd=REPO_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-evaluation.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.evaluation3_ecc_verifier import (
    EccDecisionThresholds,
    prepare_overlap_audit_ecc,
    finalize_overlap_audit,
)

PHASH_THRESHOLD = 4
THRESHOLDS = EccDecisionThresholds()
print('BRANCH:', BRANCH)
print('THRESHOLDS:', THRESHOLDS)


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
E3_DIR = DRIVE_ROOT / 'EVALUATION3'
E3_ARCHIVE = E3_DIR / 'outfit.zip'
CMT_FILE = E3_DIR / 'Cmt_ALL_20190325.xlsx'
ATTRIBUTE_FILE = E3_DIR / 'Attribute_ALL_UBSGsimple.xlsx'
SCORER_DIR = DRIVE_ROOT / 'scorer_ready_v2'
OUTPUT_DIR = DRIVE_ROOT / 'evaluation3_overlap_ecc_verifier_v0' / 'ecc_consensus_trial'

LOCAL_E3_DIR = Path('/content/evaluation3_ecc_verifier_v0')
EXTRACT_DONE = LOCAL_E3_DIR / '.extract_complete'

def locate_extracted_root():
    for candidate in [LOCAL_E3_DIR / 'outfit', LOCAL_E3_DIR / 'Outfits', LOCAL_E3_DIR]:
        if candidate.is_dir() and any(child.is_dir() for child in list(candidate.iterdir())[:50]):
            return candidate
    return None

if not E3_ARCHIVE.is_file():
    raise FileNotFoundError(f'Thiếu E3 ZIP: {E3_ARCHIVE}')
if not EXTRACT_DONE.is_file():
    if LOCAL_E3_DIR.exists():
        shutil.rmtree(LOCAL_E3_DIR)
    LOCAL_E3_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {E3_ARCHIVE} -> {LOCAL_E3_DIR} ...')
    shutil.unpack_archive(str(E3_ARCHIVE), str(LOCAL_E3_DIR), format='zip')
    EXTRACT_DONE.write_text('ok', encoding='utf-8')
else:
    print('Using existing local E3 extraction:', LOCAL_E3_DIR)

E3_ROOT = locate_extracted_root()
TRAIN_FILE = SCORER_DIR / 'scorer_ready_v2_train.jsonl'
VALID_FILE = SCORER_DIR / 'scorer_ready_v2_valid.jsonl'
TEST_FILE = SCORER_DIR / 'scorer_ready_v2_test.jsonl'
required = [CMT_FILE, ATTRIBUTE_FILE, TRAIN_FILE, VALID_FILE, TEST_FILE]
INPUTS_READY = E3_ROOT is not None and all(path.is_file() for path in required)
print('E3_ROOT:', E3_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('INPUTS:', 'OK' if INPUTS_READY else 'CHƯA ĐỦ')
if not INPUTS_READY:
    for path in required:
        if not path.is_file(): print(' - missing:', path)


## Run audit

Khác các notebook cũ, mỗi lần prepare sẽ **xóa riêng `manual_review_previews/` cũ trong output folder** trước khi chạy để tránh stale `PAIRxxxx.jpg`. Manual candidates của outfit đã có một auto-DUP cũng bị suppress vì outfit đó đã contaminated rồi.


In [ ]:
summary = None
prepare_paths = {}
if not INPUTS_READY:
    print('SKIPPED')
else:
    development_splits = {'train': TRAIN_FILE, 'valid': VALID_FILE, 'test': TEST_FILE}
    summary, prepare_paths = prepare_overlap_audit_ecc(
        evaluation3_root=E3_ROOT,
        development_split_paths=development_splits,
        output_dir=OUTPUT_DIR,
        polyvore_hf_dataset='codewaly/polyvore1000',
        annotations_path=CMT_FILE,
        annotation_sheet='CMT',
        metadata_path=ATTRIBUTE_FILE,
        metadata_sheet='Num',
        model_development_splits={'train', 'valid'},
        phash_threshold=PHASH_THRESHOLD,
        verifier_size=256,
        thresholds=THRESHOLDS,
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print('\nOutputs:')
    for name, path in prepare_paths.items(): print(f'- {name}: {path}')
    print('\nQuan trọng để calibration: evaluation3_candidate_evidence.csv')


## Manual review / calibration

Xem `manual_review_previews/` + `evaluation3_manual_review_BLIND.xlsx`. File `evaluation3_candidate_evidence.csv` chứa score của **mọi pHash candidate**, kể cả auto-DUP và NON, để sau này kiểm tra distribution thay vì tiếp tục đoán threshold.

Manual review cuối cùng vẫn chỉ dùng `DUPLICATE` hoặc `NON_DUPLICATE`.


In [ ]:
final_summary = None
final_paths = {}
if not INPUTS_READY:
    print('SKIPPED')
else:
    manual_xlsx = OUTPUT_DIR / 'evaluation3_manual_review_BLIND.xlsx'
    final_summary, final_paths = finalize_overlap_audit(
        output_dir=OUTPUT_DIR,
        manual_labels_path=manual_xlsx,
        model_development_splits={'train', 'valid'},
    )
    print(json.dumps(final_summary, ensure_ascii=False, indent=2))
    print('NOTE: PASS ở đây không có nghĩa ECC thresholds đã frozen/calibrated.')
